# Hệ Thống Dự Báo Thời Tiết — Apache Spark + MLlib

**Dataset:** **WEATHER-5K** — 5,672 trạm thời tiết toàn cầu, hourly 2014–2023.

| Bài toán | Target | Metric |
|----------|--------|--------|
| Classification | `Rain_next` — giờ sau có mưa không (Dew Point Depression < 2.5°C) | Accuracy, F1, AUC |
| Regression | `Temp_next` — nhiệt độ giờ sau (°C) | RMSE, MAE, R² |

**Checkpoint:** Lần 1 train từ đầu + lưu (`checkpoints_weather5k/`). Lần 2+ tự load. Train lại → `FORCE_RETRAIN = True`

In [1]:
import os, shutil, ctypes, json, random

HADOOP_DIR = os.path.join(r"d:\DeepLearning\Deepfake_Detection\weather_forecast", "hadoop")
os.environ["HADOOP_HOME"] = HADOOP_DIR
os.environ["PATH"] = os.path.join(HADOOP_DIR, "bin") + ";" + os.environ.get("PATH", "")
ctypes.windll.kernel32.SetEnvironmentVariableW("HADOOP_HOME", HADOOP_DIR)

VENV_PYTHON = os.path.join(r"d:\DeepLearning\Deepfake_Detection\weather_forecast\venv", "Scripts", "python.exe")
os.environ["PYSPARK_PYTHON"] = VENV_PYTHON
os.environ["PYSPARK_DRIVER_PYTHON"] = VENV_PYTHON
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator, MulticlassClassificationEvaluator, RegressionEvaluator,
)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# CONFIG
# ============================================================
PROJECT_DIR = r"d:\DeepLearning\Deepfake_Detection\weather_forecast"
DATA_DIR = os.path.join(PROJECT_DIR, "data")

WEATHER5K_DIR = os.path.join(DATA_DIR, "WEATHER-5K")
STATIONS_DIR = os.path.join(WEATHER5K_DIR, "global_weather_stations")
META_INFO_FILE = os.path.join(WEATHER5K_DIR, "meta_info.json")

# So tram sample (giam de chay nhanh, tang de chinh xac hon)
NUM_STATIONS_SAMPLE = 800
MIN_VALID_PERCENT = 0.8

# Nguong Dew Point Depression de phan loai mua
DEW_POINT_DEPRESSION_THRESHOLD = 2.5

CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints_weather5k")
MODELS_DIR = os.path.join(CHECKPOINT_DIR, "models")
TRAIN_CKPT = os.path.join(CHECKPOINT_DIR, "train.parquet")
TEST_CKPT = os.path.join(CHECKPOINT_DIR, "test.parquet")

NUMERIC_COLS = ["Temperature", "DewPoint", "Pressure", "WindSpeed", "WindDirection"]
CATEGORICAL_COLS = ["StationID"]

FORCE_RETRAIN = True  # True = xoa checkpoint, train lai tu dau
SEED = 42
# MLlib RF: giam tai heap (hang trieu dong); tang neu may du RAM
RF_NUM_TREES = 40
RF_MAX_DEPTH = 8
RF_MAX_BINS = 24
RF_SUBSAMPLING = 0.5

if FORCE_RETRAIN and os.path.exists(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)
    print("[FORCE] Xoa checkpoint. Train lai tu dau.")

os.makedirs(MODELS_DIR, exist_ok=True)
SKIP_TO_TRAIN = False

# Helpers
def save_df(df, path):
    if os.path.exists(path): shutil.rmtree(path)
    df.write.parquet(path)
    print(f"  [SAVE] {path}")

def load_df(path):
    if os.path.exists(path):
        d = spark.read.parquet(path)
        print(f"  [LOAD] {path} ({d.count():,} dong)")
        return d
    return None

def save_model(model, task, name):
    p = os.path.join(MODELS_DIR, f"{task}_{name}")
    if os.path.exists(p): shutil.rmtree(p)
    model.save(p)
    print(f"  [SAVE] Model -> {p}")

def load_model(task, name):
    p = os.path.join(MODELS_DIR, f"{task}_{name}")
    if os.path.exists(p):
        m = PipelineModel.load(p)
        print(f"  [LOAD] Model <- {p}")
        return m
    return None

print("Dataset   : WEATHER-5K")
print(f"Stations  : {NUM_STATIONS_SAMPLE} (sample)")
print(f"DPD thresh: {DEW_POINT_DEPRESSION_THRESHOLD}°C")
print(f"Checkpoint: {CHECKPOINT_DIR}")

[FORCE] Xoa checkpoint. Train lai tu dau.
Dataset   : WEATHER-5K
Stations  : 1000 (sample)
DPD thresh: 2.5°C
Checkpoint: d:\DeepLearning\Deepfake_Detection\weather_forecast\checkpoints_weather5k


In [3]:
# Spark Session
spark = SparkSession.builder \
    .appName("WeatherForecast_WEATHER5K") \
    .master("local[4]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.network.timeout", "600s") \
    .config("spark.driver.extraJavaOptions",
            f"-Dhadoop.home.dir={os.environ['HADOOP_HOME']}") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

Spark 3.5.4 | UI: http://127.0.0.1:4040


In [4]:
# ============================================================
# CHECK CHECKPOINT — co train/test san chua?
# ============================================================
SKIP_TO_TRAIN = False
train_ckpt = load_df(TRAIN_CKPT)
test_ckpt = load_df(TEST_CKPT)

if train_ckpt is not None and test_ckpt is not None:
    SKIP_TO_TRAIN = True
    print("\n>>> DA CO CHECKPOINT. Skip load/preprocessing/FE.")
    print(">>> Chay thang toi buoc train model.")
else:
    print("Chua co checkpoint. Bat dau tu CSV...")

Chua co checkpoint. Bat dau tu CSV...


## 1. Load dữ liệu WEATHER-5K

Đọc `meta_info.json` → lọc trạm hợp lệ (`valid_percent ≥ 0.8`) → sample N trạm → load CSV → filter `MASK = [1 1 1 1 1]` → union

In [ ]:
if not SKIP_TO_TRAIN:
    # Load metadata
    with open(META_INFO_FILE, "r", encoding="utf-8") as f:
        meta = json.load(f)
    
    # Loc tram hop le
    valid_stations = [
        fname for fname, info in meta.items()
        if info.get("valid_percent", 0) >= MIN_VALID_PERCENT
    ]
    print(f"Tram hop le (valid >= {MIN_VALID_PERCENT}): {len(valid_stations)}/{len(meta)}")
    
    # Sample
    random.seed(SEED)
    if NUM_STATIONS_SAMPLE < len(valid_stations):
        selected = random.sample(valid_stations, NUM_STATIONS_SAMPLE)
    else:
        selected = valid_stations
    print(f"Selected: {len(selected)} tram")
    
    # Schema
    SCHEMA = StructType([
        StructField("DATE", StringType(), True),
        StructField("LONGITUDE", DoubleType(), True),
        StructField("LATITUDE", DoubleType(), True),
        StructField("TMP", DoubleType(), True),
        StructField("DEW", DoubleType(), True),
        StructField("WND_ANGLE", DoubleType(), True),
        StructField("WND_RATE", DoubleType(), True),
        StructField("SLP", DoubleType(), True),
        StructField("MASK", StringType(), True),
        StructField("TIME_DIFF", DoubleType(), True),
    ])
    
    # Load tung file
    dfs = []
    for fname in selected:
        fpath = os.path.join(STATIONS_DIR, fname)
        if not os.path.exists(fpath):
            continue
        station_id = fname.replace(".csv", "")
        sdf = (
            spark.read.csv(fpath, header=True, schema=SCHEMA)
            .withColumn("StationID", F.lit(station_id))
        )
        dfs.append(sdf)
    print(f"Loaded: {len(dfs)} files")
    
    # Union
    df_raw = dfs[0]
    for d in dfs[1:]:
        df_raw = df_raw.unionByName(d)
    
    # Filter MASK
    before = df_raw.count()
    df_raw = df_raw.filter(F.col("MASK") == "[1 1 1 1 1]")
    after = df_raw.count()
    print(f"MASK filter: {before:,} -> {after:,} (loai {before-after:,} invalid)")
    
    # Rename columns
    df_raw = (
        df_raw
        .withColumnRenamed("DATE", "datetime")
        .withColumnRenamed("TMP", "Temperature")
        .withColumnRenamed("DEW", "DewPoint")
        .withColumnRenamed("WND_ANGLE", "WindDirection")
        .withColumnRenamed("WND_RATE", "WindSpeed")
        .withColumnRenamed("SLP", "Pressure")
    )
    
    # Drop columns khong can
    df_raw = df_raw.drop("MASK", "TIME_DIFF", "LONGITUDE", "LATITUDE")
    
    n = df_raw.count()
    stations = df_raw.select("StationID").distinct().count()
    print(f"\nWEATHER-5K: {n:,} dong, {stations} tram")
    df_raw.show(5, truncate=False)
else:
    print(">>> Skip (checkpoint loaded)")

Tram hop le (valid >= 0.8): 5672/5672
Selected: 1000 tram
Loaded: 1000 files


## 2. EDA — Khám phá dữ liệu

In [ ]:
if not SKIP_TO_TRAIN:
    print("MISSING VALUES")
    total = df_raw.count()
    cols_miss = ["Temperature", "DewPoint", "Pressure", "WindSpeed", "WindDirection"]
    for c in cols_miss:
        if c in df_raw.columns:
            m = df_raw.filter(F.col(c).isNull()).count()
            if m > 0:
                print(f"  {c:<20s}: {m:>10,} ({m/total*100:.1f}%)")

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    
    temp_pd = df_raw.select("Temperature").dropna().limit(500000).toPandas()
    axes[0].hist(temp_pd["Temperature"], bins=60, color="#e74c3c", edgecolor="white")
    axes[0].set_xlabel("°C")
    axes[0].set_title("Phân bố Nhiệt độ")
    
    dew_pd = df_raw.select("DewPoint").dropna().limit(500000).toPandas()
    axes[1].hist(dew_pd["DewPoint"], bins=60, color="#3498db", edgecolor="white")
    axes[1].set_xlabel("°C")
    axes[1].set_title("Phân bố Dew Point")
    
    dpd_pd = df_raw.withColumn("DPD", F.col("Temperature") - F.col("DewPoint")).select("DPD").dropna().limit(500000).toPandas()
    axes[2].hist(dpd_pd["DPD"], bins=60, color="#2ecc71", edgecolor="white")
    axes[2].axvline(DEW_POINT_DEPRESSION_THRESHOLD, color="red", linestyle="--", linewidth=2, label=f"Rain threshold ({DEW_POINT_DEPRESSION_THRESHOLD}°C)")
    axes[2].set_xlabel("Temperature - DewPoint (°C)")
    axes[2].set_title("Dew Point Depression")
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print(">>> Skip EDA (checkpoint loaded)")

## 3. Preprocessing

1. Parse datetime
2. Dew Point Depression → Rain label (DPD < 2.5°C → Rain = 1)
3. Target Rain_next, Temp_next (lead 1h theo trạm)
4. Fill median cho NUMERIC_COLS

In [ ]:
if not SKIP_TO_TRAIN:
    df = df_raw

    # Parse datetime
    df = df.withColumn("datetime_ts", F.to_timestamp("datetime"))
    df = df.filter(F.col("datetime_ts").isNotNull())
    print("Parsed datetime")

    # Dew Point Depression + Rain label
    df = df.withColumn("DewPointDepression", F.col("Temperature") - F.col("DewPoint"))
    df = df.withColumn(
        "Rain",
        F.when(F.col("DewPointDepression") < F.lit(DEW_POINT_DEPRESSION_THRESHOLD), 1.0).otherwise(0.0)
    )
    
    st = df.select(F.count(F.lit(1)).alias("n"), F.sum("Rain").alias("rain_sum")).first()
    total, rain_count = int(st["n"]), int(st["rain_sum"])
    print(f"Rain label (DPD < {DEW_POINT_DEPRESSION_THRESHOLD}°C): {rain_count:,}/{total:,} ({rain_count/total*100:.1f}%) positive")

    # Targets (lead theo StationID)
    w = Window.partitionBy("StationID").orderBy("datetime_ts")
    df = df.withColumn("Rain_next", F.lead("Rain", 1).over(w))
    df = df.withColumn("Temp_next", F.lead("Temperature", 1).over(w))
    df = df.withColumn("label_clf", F.col("Rain_next"))
    print("Targets: Rain_next (clf), Temp_next (reg)")

    # Drop null targets
    df = df.dropna(subset=["Rain_next", "Temp_next"])
    print("Drop null targets")

    # Fill median
    df = df.cache()
    medians = {}
    for c in NUMERIC_COLS:
        if c in df.columns:
            med = df.approxQuantile(c, [0.5], 0.05)
            if med:
                medians[c] = med[0]
    if medians:
        df = df.fillna(medians)
    print(f"Filled median: {list(medians.keys())}")

    n = df.count()
    print(f"\nSau preprocessing: {n:,} dong, {len(df.columns)} cot")
    df.groupBy("label_clf").count().orderBy("label_clf").show()
else:
    print(">>> Skip preprocessing (checkpoint loaded)")

## 4. Feature Engineering

| Loại | Features |
|------|----------|
| **Time** | Hour, Month, DayOfWeek, DayOfYear + cyclic sin/cos |
| **Lag** | Temp 1/3/6/24h, DewPoint/Pressure/WindSpeed 1h, Rain 1/3h |
| **Rolling** | Trung bình 6h và 24h: Temp, DewPoint, Pressure. Tổng Rain 6h/24h |
| **Derived** | TempChange_1h, DewPointChange_1h, PressureChange_1h, TempDewPoint |

In [ ]:
if not SKIP_TO_TRAIN:
    # TIME FEATURES
    df = df.withColumn("Hour", F.hour("datetime_ts"))
    df = df.withColumn("Month", F.month("datetime_ts"))
    df = df.withColumn("DayOfWeek", F.dayofweek("datetime_ts"))
    df = df.withColumn("DayOfYear", F.dayofyear("datetime_ts"))
    df = df.withColumn("Hour_sin", F.sin(2*3.14159*F.col("Hour")/24))
    df = df.withColumn("Hour_cos", F.cos(2*3.14159*F.col("Hour")/24))
    df = df.withColumn("Month_sin", F.sin(2*3.14159*F.col("Month")/12))
    df = df.withColumn("Month_cos", F.cos(2*3.14159*F.col("Month")/12))
    print("+ Time features")

    # LAG FEATURES
    w = Window.partitionBy("StationID").orderBy("datetime_ts")
    df = df.withColumn("Temp_lag1", F.lag("Temperature", 1).over(w))
    df = df.withColumn("Temp_lag3", F.lag("Temperature", 3).over(w))
    df = df.withColumn("Temp_lag6", F.lag("Temperature", 6).over(w))
    df = df.withColumn("Temp_lag24", F.lag("Temperature", 24).over(w))
    df = df.withColumn("DewPoint_lag1", F.lag("DewPoint", 1).over(w))
    df = df.withColumn("Pressure_lag1", F.lag("Pressure", 1).over(w))
    df = df.withColumn("WindSpeed_lag1", F.lag("WindSpeed", 1).over(w))
    df = df.withColumn("Rain_lag1", F.lag("Rain", 1).over(w))
    df = df.withColumn("Rain_lag3", F.lag("Rain", 3).over(w))
    print("+ Lag features (1/3/6/24h)")

    # ROLLING FEATURES
    w6 = Window.partitionBy("StationID").orderBy("datetime_ts").rowsBetween(-6, -1)
    w24 = Window.partitionBy("StationID").orderBy("datetime_ts").rowsBetween(-24, -1)
    df = df.withColumn("Temp_roll6h", F.avg("Temperature").over(w6))
    df = df.withColumn("Temp_roll24h", F.avg("Temperature").over(w24))
    df = df.withColumn("DewPoint_roll6h", F.avg("DewPoint").over(w6))
    df = df.withColumn("Pressure_roll6h", F.avg("Pressure").over(w6))
    df = df.withColumn("Rain_sum6h", F.sum("Rain").over(w6))
    df = df.withColumn("Rain_sum24h", F.sum("Rain").over(w24))
    print("+ Rolling features (6h/24h)")

    # DERIVED FEATURES
    df = df.withColumn("TempChange_1h", F.col("Temperature") - F.col("Temp_lag1"))
    df = df.withColumn("DewPointChange_1h", F.col("DewPoint") - F.col("DewPoint_lag1"))
    df = df.withColumn("PressureChange_1h", F.col("Pressure") - F.col("Pressure_lag1"))
    df = df.withColumn("TempDewPoint", F.col("Temperature") - F.col("DewPoint"))
    print("+ Derived features")

    # Drop NaN from lag/rolling
    before = df.count()
    df = df.dropna()
    after = df.count()
    print(f"\nDrop NaN: {before:,} -> {after:,} (giam {before-after:,})")
    print(f"Total columns: {len(df.columns)}")
else:
    print(">>> Skip FE (checkpoint loaded)")

## 5. ML Pipeline + Split Train/Test

In [ ]:
# Pipeline stages
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in CATEGORICAL_COLS]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_vec", handleInvalid="keep") for c in CATEGORICAL_COLS]

feature_cols = NUMERIC_COLS + [
    "DewPointDepression", "Rain",
    "Hour", "Month", "DayOfWeek", "DayOfYear",
    "Hour_sin", "Hour_cos", "Month_sin", "Month_cos",
    "Temp_lag1", "Temp_lag3", "Temp_lag6", "Temp_lag24",
    "DewPoint_lag1", "Pressure_lag1", "WindSpeed_lag1",
    "Rain_lag1", "Rain_lag3",
    "Temp_roll6h", "Temp_roll24h", "DewPoint_roll6h", "Pressure_roll6h",
    "Rain_sum6h", "Rain_sum24h",
    "TempChange_1h", "DewPointChange_1h", "PressureChange_1h",
    "TempDewPoint",
] + [f"{c}_vec" for c in CATEGORICAL_COLS]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw", handleInvalid="skip")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)
base_stages = indexers + encoders + [assembler, scaler]

print(f"Feature cols: {len(feature_cols)}")
print(f"Base stages : {len(base_stages)}")

In [ ]:
# Split train/test (co checkpoint)
if SKIP_TO_TRAIN:
    train_df = train_ckpt
    test_df = test_ckpt
    print("Da load train/test tu checkpoint.")
else:
    train_df, test_df = df.randomSplit([0.8, 0.2], seed=SEED)
    save_df(train_df, TRAIN_CKPT)
    save_df(test_df, TEST_CKPT)

train_df.cache()
test_df.cache()
tr = train_df.count()
te = test_df.count()
print(f"\nTrain: {tr:,} | Test: {te:,} | Ratio: {tr/(tr+te):.0%}/{te/(tr+te):.0%}")

## 6. Classification — Dự đoán Rain_next (giờ tới có mưa?)

| Model | Mô tả |
|-------|--------|
| **Logistic Regression** | Baseline, nhanh, tuyến tính |
| **Random Forest** | Ensemble bagging, non-linear |
| **GBT Classifier** | Boosting tuần tự, hiệu suất cao |

In [ ]:
def eval_clf(preds, label_col="label_clf"):
    b = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="rawPrediction")
    m = MulticlassClassificationEvaluator(labelCol=label_col, predictionCol="prediction")
    auc = b.evaluate(preds, {b.metricName: "areaUnderROC"})
    r = {}
    for mn in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
        m.setMetricName(mn); r[mn] = m.evaluate(preds)
    return {"Accuracy": r["accuracy"], "Precision": r["weightedPrecision"],
            "Recall": r["weightedRecall"], "F1": r["f1"], "AUC": auc}

def print_cm(preds, label_col="label_clf"):
    tp = preds.filter((F.col(label_col)==1)&(F.col("prediction")==1)).count()
    tn = preds.filter((F.col(label_col)==0)&(F.col("prediction")==0)).count()
    fp = preds.filter((F.col(label_col)==0)&(F.col("prediction")==1)).count()
    fn = preds.filter((F.col(label_col)==1)&(F.col("prediction")==0)).count()
    print(f"  Confusion Matrix:  TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}")

In [ ]:
LABEL_CLF = "label_clf"

clf_defs = {
    "Logistic Regression": ("clf_lr", LogisticRegression(
        featuresCol="features", labelCol=LABEL_CLF, maxIter=100, regParam=0.01)),
    "Random Forest": ("clf_rf", RandomForestClassifier(
        featuresCol="features", labelCol=LABEL_CLF,
        numTrees=RF_NUM_TREES, maxDepth=RF_MAX_DEPTH, maxBins=RF_MAX_BINS,
        subsamplingRate=RF_SUBSAMPLING, seed=SEED)),
    "GBT Classifier": ("clf_gbt", GBTClassifier(
        featuresCol="features", labelCol=LABEL_CLF, maxIter=50, maxDepth=5, seed=SEED)),
}

clf_results = {}
for name, (key, estimator) in clf_defs.items():
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")

    fitted = load_model("clf", key)
    if fitted is None:
        print(f"  Training...")
        pipeline = Pipeline(stages=base_stages + [estimator])
        fitted = pipeline.fit(train_df)
        save_model(fitted, "clf", key)
    else:
        print(f"  Loaded from checkpoint.")

    print("  Scoring test set (RF/GBT can take several minutes)...")
    preds = fitted.transform(test_df)
    metrics = eval_clf(preds, LABEL_CLF)
    clf_results[name] = metrics
    for k, v in metrics.items():
        print(f"  {k:<12s}: {v:.4f}")
    print_cm(preds, LABEL_CLF)

print("\n\nClassification DONE!")

In [ ]:
# So sanh Classification
print("="*75)
print("SO SANH CLASSIFICATION")
print("="*75)
hdr = f"{'Model':<25s}{'Accuracy':>10s}{'Precision':>10s}{'Recall':>10s}{'F1':>10s}{'AUC':>10s}"
print(hdr); print("-"*75)
for n, m in clf_results.items():
    print(f"{n:<25s}" + "".join(f"{v:>10.4f}" for v in m.values()))
best = max(clf_results, key=lambda x: clf_results[x]["AUC"])
print(f"\nBest (AUC): {best}")

fig, ax = plt.subplots(figsize=(12, 5))
models = list(clf_results.keys())
x = np.arange(len(models))
w = 0.15
mnames = ["Accuracy","Precision","Recall","F1","AUC"]
colors = ["#3498db","#2ecc71","#e74c3c","#f39c12","#9b59b6"]
for i,(mn,c) in enumerate(zip(mnames,colors)):
    vals = [clf_results[m][mn] for m in models]
    ax.bar(x+i*w, vals, w, label=mn, color=c)
ax.set_xticks(x+w*2); ax.set_xticklabels(models)
ax.set_ylabel("Score"); ax.set_ylim(0.5,1.0)
ax.legend(loc="lower right"); ax.set_title("Classification Comparison")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Regression — Dự đoán Temp_next (nhiệt độ giờ tới °C)

| Metric | Ý nghĩa |
|--------|----------|
| **RMSE** | Sai số trung bình (°C) — thấp = tốt |
| **MAE** | Sai số tuyệt đối trung bình |
| **R²** | Giải thích được bao nhiêu % variance — gần 1 = tốt |

In [ ]:
LABEL_REG = "label_reg"
train_reg = train_df.withColumn(LABEL_REG, F.col("Temp_next").cast("double"))
test_reg = test_df.withColumn(LABEL_REG, F.col("Temp_next").cast("double"))

def eval_reg(preds, label_col="label_reg"):
    e = RegressionEvaluator(labelCol=label_col, predictionCol="prediction")
    e.setMetricName("rmse"); rmse = e.evaluate(preds)
    e.setMetricName("mae");  mae = e.evaluate(preds)
    e.setMetricName("r2");   r2 = e.evaluate(preds)
    return {"RMSE": rmse, "MAE": mae, "R2": r2}

reg_defs = {
    "Linear Regression": ("reg_linear", LinearRegression(
        featuresCol="features", labelCol=LABEL_REG, maxIter=100, regParam=0.01)),
    "Random Forest": ("reg_rf", RandomForestRegressor(
        featuresCol="features", labelCol=LABEL_REG,
        numTrees=RF_NUM_TREES, maxDepth=RF_MAX_DEPTH, maxBins=RF_MAX_BINS,
        subsamplingRate=RF_SUBSAMPLING, seed=SEED)),
    "GBT Regressor": ("reg_gbt", GBTRegressor(
        featuresCol="features", labelCol=LABEL_REG, maxIter=50, maxDepth=5, seed=SEED)),
}

reg_results = {}
for name, (key, estimator) in reg_defs.items():
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")

    fitted = load_model("reg", key)
    if fitted is None:
        print(f"  Training...")
        pipeline = Pipeline(stages=base_stages + [estimator])
        fitted = pipeline.fit(train_reg)
        save_model(fitted, "reg", key)
    else:
        print(f"  Loaded from checkpoint.")

    print("  Scoring test set (RF/GBT can take several minutes)...")
    preds = fitted.transform(test_reg)
    metrics = eval_reg(preds, LABEL_REG)
    reg_results[name] = metrics
    for k, v in metrics.items():
        print(f"  {k:<6s}: {v:.4f}")

print("\n\nRegression DONE!")

In [ ]:
# So sanh Regression
print("="*55)
print("SO SANH REGRESSION")
print("="*55)
hdr = f"{'Model':<25s}{'RMSE':>10s}{'MAE':>10s}{'R2':>10s}"
print(hdr); print("-"*55)
for n, m in reg_results.items():
    print(f"{n:<25s}" + "".join(f"{v:>10.4f}" for v in m.values()))
best = min(reg_results, key=lambda x: reg_results[x]["RMSE"])
print(f"\nBest (RMSE): {best}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models = list(reg_results.keys())
colors = ["#3498db","#2ecc71","#e74c3c"]
for i,(metric,title) in enumerate([("RMSE","RMSE (low=good)"),("MAE","MAE (low=good)"),("R2","R² (high=good)")]):
    vals = [reg_results[m][metric] for m in models]
    axes[i].bar(models, vals, color=colors)
    axes[i].set_title(title); axes[i].set_ylabel(metric)
    for j,v in enumerate(vals):
        axes[i].text(j, v+0.01, f"{v:.3f}", ha="center", fontsize=10)
plt.tight_layout(); plt.show()

## 8. Tổng kết

In [ ]:
print("="*75)
print("TONG KET")
print("="*75)

print("\n--- CLASSIFICATION: Rain_next ---")
hdr = f"{'Model':<25s}{'Accuracy':>10s}{'Precision':>10s}{'Recall':>10s}{'F1':>10s}{'AUC':>10s}"
print(hdr); print("-"*75)
for n, m in clf_results.items():
    print(f"{n:<25s}" + "".join(f"{v:>10.4f}" for v in m.values()))

print(f"\n--- REGRESSION: Temp_next ---")
hdr = f"{'Model':<25s}{'RMSE':>10s}{'MAE':>10s}{'R2':>10s}"
print(hdr); print("-"*55)
for n, m in reg_results.items():
    print(f"{n:<25s}" + "".join(f"{v:>10.4f}" for v in m.values()))

# Save metrics JSON cho dashboard
metrics_out = {
    "classification": clf_results,
    "regression": reg_results,
    "train_count": train_df.count(),
    "test_count": test_df.count(),
}
metrics_path = os.path.join(CHECKPOINT_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"\nMetrics saved -> {metrics_path}")
print("Dashboard doc file nay de hien thi ket qua.")

print(f"\nCheckpoints: {CHECKPOINT_DIR}")
print("Lan chay tiep theo se load tu checkpoint (khong train lai).")
print("Muon train lai: FORCE_RETRAIN = True")

spark.stop()
print("\nSpark stopped. DONE!")